# 02 — Tiền Xử Lý Video (Data Cleaning & Imputation)

Notebook này thực hiện quy trình **tiền xử lý video** (preprocessing) cho
dữ liệu ngôn ngữ ký hiệu Việt Nam (VSL-400), bao gồm 2 giai đoạn chính:

1. **Temporal Boundary Localization (TBL):** Xác định ranh giới thời gian
   của hành động ký hiệu, loại bỏ các khung hình tĩnh đầu/cuối.
2. **Spatial Crop & Resize:** Cắt vùng đầu-vai-eo và nén về kích thước
   chuẩn `224x224` pixel.

> **Thuật toán TBL** dựa trên góc khuỷu tay (elbow angle) được trích xuất
> từ MediaPipe Pose. Nếu góc khuỷu < 160°, frame được xem là "active"
> (người đang thực hiện ký hiệu).

---
## 1. Import Thư Viện & Cấu Hình MediaPipe

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import os
import json
import math
import glob
import concurrent.futures
from tqdm import tqdm

# Khởi tạo MediaPipe Pose
mp_pose = mp.solutions.pose

# Danh sách 6 điểm mốc liên quan đến tay/vai
LANDMARKS = {
    "LShoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "RShoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "LElbow":    mp_pose.PoseLandmark.LEFT_ELBOW,
    "RElbow":    mp_pose.PoseLandmark.RIGHT_ELBOW,
    "LWrist":    mp_pose.PoseLandmark.LEFT_WRIST,
    "RWrist":    mp_pose.PoseLandmark.RIGHT_WRIST,
}

print("Đã khởi tạo MediaPipe Pose và cấu hình landmarks.")

---
## 2. Thuật Toán Định Vị Thời Gian (TBL - Temporal Boundary Localization)

Các hàm bổ trợ tính toán góc khuỷu tay và xác định trạng thái **active/inactive**
của người ký dựa trên vị trí các khớp vai và cổ tay.

### Nguyên lý hoạt động:
- Tính góc giữa 3 điểm: **Vai → Khuỷu tay → Cổ tay**
- Nếu góc trung bình < `θ = 160°` → Frame **active** (đang ký)
- Nếu góc ≥ `θ` → Frame **inactive** (tay duỗi thẳng, không ký)

### Các tham số:
| Tham số | Giá trị | Ý nghĩa |
|---------|---------|---------|
| `theta` | 160° | Ngưỡng góc khuỷu tay |
| `vis_th` | 0.6 | Ngưỡng visibility tối thiểu |
| `point_th` | 0.5 | Ngưỡng visibility cho từng điểm |
| `t_min` | 0.67s | Thời lượng tối thiểu của đoạn active |
| `max_gap` | 0.8s | Khoảng cách tối đa để gộp 2 đoạn |
| `padding` | 0.4s | Padding thêm trước/sau đoạn active |

In [ ]:
def _angle_2d(a, b, c):
    """
    Tính góc ABC (độ) trong mặt phẳng 2D.
    
    Parameters
    ----------
    a, b, c : tuple(float, float)
        Tọa độ (x, y) của 3 điểm A, B, C.
    
    Returns
    -------
    int
        Góc tại B (độ), luôn trả về int.
    """
    ba_x, ba_y = a[0] - b[0], a[1] - b[1]
    bc_x, bc_y = c[0] - b[0], c[1] - b[1]

    norm_ba = math.hypot(ba_x, ba_y)
    norm_bc = math.hypot(bc_x, bc_y)
    if norm_ba == 0 or norm_bc == 0:
        return 0

    cos_val = (ba_x * bc_x + ba_y * bc_y) / (norm_ba * norm_bc)
    cos_val = max(-1.0, min(1.0, cos_val))
    return int(round(math.degrees(math.acos(cos_val))))


def frame_active_from_landmarks(lm, theta=160, vis_th=0.6, point_th=0.5):
    """
    Kiểm tra frame active bằng cách tính góc khuỷu tay.
    
    Parameters
    ----------
    lm : list
        Danh sách landmarks từ MediaPipe Pose.
    theta : int
        Ngưỡng góc khuỷu tay (độ). Frame active khi góc < theta.
    vis_th : float
        Ngưỡng visibility tối thiểu cho tất cả landmarks.
    point_th : float
        Ngưỡng visibility cho từng điểm riêng lẻ.
    
    Returns
    -------
    int
        1 nếu frame active, 0 nếu inactive.
    """
    # Trả về 0 (inactive) nếu bất kỳ điểm nào ở tay/vai bị mờ/khuất
    if any(float(lm[idx].visibility) < vis_th for idx in LANDMARKS.values()):
        return 0

    elbow_pairs = [
        (mp_pose.PoseLandmark.LEFT_SHOULDER,  mp_pose.PoseLandmark.LEFT_ELBOW,  mp_pose.PoseLandmark.LEFT_WRIST),
        (mp_pose.PoseLandmark.RIGHT_SHOULDER, mp_pose.PoseLandmark.RIGHT_ELBOW, mp_pose.PoseLandmark.RIGHT_WRIST),
    ]

    angles = []
    for s_idx, e_idx, w_idx in elbow_pairs:
        s_lm, e_lm, w_lm = lm[s_idx], lm[e_idx], lm[w_idx]
        if min(s_lm.visibility, e_lm.visibility, w_lm.visibility) < point_th:
            continue
        angle = _angle_2d(
            (s_lm.x, s_lm.y), 
            (e_lm.x, e_lm.y), 
            (w_lm.x, w_lm.y)
        )
        if angle > 0:
            angles.append(angle)

    if not angles:
        return 0
    
    # Lấy góc trung bình của cả 2 tay
    elbow_angle = int(round(sum(angles) / len(angles)))
    return 1 if elbow_angle < theta else 0

---
## 3. Hàm Tiền Xử Lý Video Đơn Lẻ (2-Pass Pipeline)

Tích hợp quy trình 2 bước:

### Pass 1 — Temporal Boundary Localization
- Quét luồng video, đánh giá từng frame active/inactive
- Trích xuất các phân đoạn chuyển động liên tục
- Gộp các đoạn quá gần nhau (< 0.8s) để tránh nhiễu
- Lọc các đoạn đạt chuẩn thời lượng tối thiểu (≥ 0.67s)
- Loại bỏ video có nhiều hơn 1 đoạn active (nghi ngờ lỗi/nhiễu)

### Pass 2 — Spatial Crop & Resize
- Tìm Bounding Box chuẩn quanh cơ thể tại khung hình giữa (mid-frame)
- Box size = `shoulder_width × 3.6` (bao phủ đầu đến hông)
- Áp dụng padding 0.4s trước/sau đoạn active
- Cắt và nén video vuông 1:1 (`224×224`)

In [ ]:
# ===============================================
# BIẾN TOÀN CỤC — Detector MediaPipe dùng chung
# ===============================================
_GLOBAL_POSE_DETECTOR = None

def _get_pose_detector():
    """Singleton pattern: Tạo 1 instance MediaPipe Pose duy nhất trên mỗi tiến trình."""
    global _GLOBAL_POSE_DETECTOR
    if _GLOBAL_POSE_DETECTOR is None:
        _GLOBAL_POSE_DETECTOR = mp_pose.Pose(
            static_image_mode=False, 
            model_complexity=1
        )
    return _GLOBAL_POSE_DETECTOR


def process_single_front_video(video_id, src_video_path, output_root, 
                                theta=160, target_size=224):
    """
    Quy trình siêu tối ưu: Đọc video 1 lần vào RAM, cache tọa độ xương khớp,
    sử dụng chung một instance MediaPipe toàn cục trên mỗi CPU core.
    
    Parameters
    ----------
    video_id : str
        ID của video.
    src_video_path : str
        Đường dẫn đến file video nguồn.
    output_root : str
        Thư mục đầu ra.
    theta : int
        Ngưỡng góc khuỷu tay (mặc định: 160°).
    target_size : int
        Kích thước đầu ra vuông (mặc định: 224px).
    
    Returns
    -------
    str
        Thông báo kết quả xử lý.
    """
    try:
        cap = cv2.VideoCapture(src_video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0: 
            fps = 25.0
            
        frames = []          # Lưu toàn bộ các frame vào RAM
        s_raw = []            # Chuỗi trạng thái active/inactive
        coord_cache = []      # Cache tọa độ phục vụ tính Bounding Box
        
        # Lấy bộ detector dùng chung của CPU core hiện tại
        pose = _get_pose_detector()
        
        # ===== PASS 1: ĐỌC LUỒNG DUY NHẤT & TRÍCH XUẤT TOÀN BỘ DỮ LIỆU =====
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: 
                break
            
            frames.append(frame)
            
            result = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if not result.pose_landmarks:
                s_raw.append(0)
                coord_cache.append(None)
                continue
                
            lm = result.pose_landmarks.landmark
            is_active = frame_active_from_landmarks(lm, theta=theta)
            s_raw.append(is_active)
            
            # CACHE TỌA ĐỘ: Chỉ lưu những điểm cần thiết cho Crop
            coord_cache.append({
                'nose_x':  lm[mp_pose.PoseLandmark.NOSE].x,
                'nose_y':  lm[mp_pose.PoseLandmark.NOSE].y,
                'l_sh_x':  lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
                'r_sh_x':  lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x,
            })
        cap.release()
        
        total_frames = len(frames)
        if total_frames == 0:
            return f"⏩ Skip ID {video_id}: Video hỏng hoặc không có frame."
            
        # Trích xuất các đoạn hành động liên tục
        segments = []
        i = 0
        while i < total_frames:
            if s_raw[i] == 0:
                i += 1
                continue
            j = i
            while j < total_frames and s_raw[j] == 1: 
                j += 1
            segments.append((i, j - 1))
            i = j
            
        # Gộp các đoạn quá gần nhau (< 0.8 giây)
        max_gap_frames = int(round(fps * 0.8)) 
        merged_segments = []
        
        if segments:
            current_start, current_end = segments[0]
            for next_start, next_end in segments[1:]:
                if next_start - current_end <= max_gap_frames:
                    current_end = next_end
                else:
                    merged_segments.append((current_start, current_end))
                    current_start, current_end = next_start, next_end
            merged_segments.append((current_start, current_end))
        
        # Lọc đoạn đạt chuẩn thời lượng tối thiểu (t_min = 0.67s)
        valid_segments = []
        for start_idx, end_idx in merged_segments:
            duration = ((end_idx + 1) - start_idx) / fps
            if duration >= 0.67:
                valid_segments.append((start_idx, end_idx))
                
        if len(valid_segments) == 0:
            return f"⏩ Skip ID {video_id}: Không tìm thấy đoạn active hợp lệ."
            
        elif len(valid_segments) > 1:
            return (f"⏩ Skip ID {video_id}: Phát hiện {len(valid_segments)} "
                    f"phân đoạn (nghi ngờ lỗi/nhiễu) → Đã loại bỏ.")
        
        # Lúc này chắc chắn len(valid_segments) == 1
        start_frame, end_frame = valid_segments[0]

        # ===== PASS 2: CẮT KHÔNG GIAN (CROP) & RESIZE TỪ RAM =====
        os.makedirs(output_root, exist_ok=True)
        h, w, _ = frames[0].shape
        
        for seg_idx, (start_frame, end_frame) in enumerate(valid_segments):
            mid_frame_idx = (start_frame + end_frame) // 2
            padding_frames = int(round(fps * 0.4))
            
            # Nới rộng khoảng thời gian
            start_frame = max(0, start_frame - padding_frames)
            end_frame = min(total_frames - 1, end_frame + padding_frames)
            
            # Lấy tọa độ từ cache (đã tính ở Pass 1)
            mid_coords = coord_cache[mid_frame_idx]
            
            box_size = min(h, w)
            x1, y1 = (w - box_size) // 2, (h - box_size) // 2
            
            if mid_coords is not None:
                pixel_nose_x = int(mid_coords['nose_x'] * w)
                pixel_l_sh_x = int(mid_coords['l_sh_x'] * w)
                pixel_r_sh_x = int(mid_coords['r_sh_x'] * w)
                pixel_nose_y = int(mid_coords['nose_y'] * h)
                
                shoulder_width = abs(pixel_l_sh_x - pixel_r_sh_x)
                box_size = int(shoulder_width * 3.6) 
                box_size = min(box_size, min(h, w))
                
                x1 = pixel_nose_x - box_size // 2
                y1 = pixel_nose_y - int(shoulder_width * 0.6)
            
            # Thuật toán chống tràn viền
            if y1 < 0: y1 = 0
            if y1 + box_size > h: y1 = h - box_size
            if x1 < 0: x1 = 0
            if x1 + box_size > w: x1 = w - box_size
            
            # Ghi video mới
            if len(valid_segments) == 1:
                out_path = os.path.join(output_root, f"{video_id}.mp4")
            else:
                out_path = os.path.join(output_root, f"{video_id}_{seg_idx:02d}.mp4")
            
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out_writer = cv2.VideoWriter(
                out_path, fourcc, fps, (target_size, target_size)
            )
            
            # Lấy trực tiếp từ mảng frames trong RAM
            for f_idx in range(start_frame, end_frame + 1):
                frame = frames[f_idx]
                cropped = frame[y1:y1+box_size, x1:x1+box_size]
                resized = cv2.resize(
                    cropped, (target_size, target_size), 
                    interpolation=cv2.INTER_AREA
                )
                out_writer.write(resized)
                
            out_writer.release()
            
        return f"Đã xử lý thành công ID: {video_id}"
    except Exception as e:
        return f"Lỗi tại ID {video_id}: {str(e)}"

In [ ]:
def video_worker(video_path):
    """
    Hàm worker trung gian cho ProcessPoolExecutor.
    
    Tự động trích xuất video_id và class_name từ đường dẫn file.
    """
    try:
        output_root = os.path.join("..", "VSL_FULL_FRONT_CROPPED_TO224x224_V2")
        parent_dir = os.path.basename(os.path.dirname(video_path))
        video_id = os.path.splitext(os.path.basename(video_path))[0]
        class_output_dir = os.path.join(output_root, parent_dir)
        
        res = process_single_front_video(video_id, video_path, class_output_dir)
        return res
    except Exception as e:
        return f"Lỗi hệ thống tại {video_path}: {str(e)}"

---
## 4. Thực Thi Tiền Xử Lý Hàng Loạt (Batch Processing)

Quét toàn bộ thư mục gốc `VSL_FULL_FRONT`, tự động áp dụng quy trình
tiền xử lý **đa nhân** (parallel processing) cho từng video.

> **Lưu ý quan trọng:**
> - Quá trình này tốn rất nhiều RAM vì mỗi video được đọc hoàn toàn vào bộ nhớ
> - Thời gian xử lý phụ thuộc vào số lượng CPU cores và kích thước dataset
> - Trên máy 8-core, ~25,000 video mất khoảng 4-8 giờ

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
INPUT_ROOT = os.path.join("..", "VSL_FULL_FRONT_RAW")
OUTPUT_ROOT = os.path.join("..", "VSL_FULL_FRONT_CROPPED_TO224x224_V2")

# Tìm tất cả video cần xử lý
all_videos = glob.glob(os.path.join(INPUT_ROOT, "*", "*.mp4"))

num_workers = os.cpu_count()

print(f"Khởi động xử lý ĐA NHÂN (Parallel Processing)...")
print(f" Số lượng nhân CPU sử dụng: {num_workers}")
print(f"Tổng số video cần xử lý: {len(all_videos)}")
print("-" * 50)

### Chạy xử lý

In [ ]:
if __name__ == "__main__" and len(all_videos) > 0:
    thong_ke = {"thanh_cong": 0, "bo_qua": 0, "loi": 0}
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(video_worker, path): path 
            for path in all_videos
        }
        
        for future in tqdm(
            concurrent.futures.as_completed(futures), 
            total=len(futures), 
            desc="Đang xử lý"
        ):
            try:
                res = future.result()
                if "" in res:
                    thong_ke["thanh_cong"] += 1
                elif "⏩ Skip" in res:
                    thong_ke["bo_qua"] += 1
                else:
                    thong_ke["loi"] += 1
            except Exception:
                thong_ke["loi"] += 1
                
    print("\n" + "=" * 15 + " BÁO CÁO KẾT QUẢ CROP VIDEO (ĐA NHÂN) " + "=" * 15)
    print(f"  Tổng số video quét:     {len(all_videos)} video")
    print(f"  Đã crop thành công:     {thong_ke['thanh_cong']} video")
    print(f"  Bị bỏ qua (quá ngắn):  {thong_ke['bo_qua']} video")
    print(f"  Bị lỗi hệ thống:       {thong_ke['loi']} video")
    print("=" * 60)
else:
    print("Không tìm thấy video nào hoặc không chạy từ __main__.")
    print("   Hãy kiểm tra lại đường dẫn INPUT_ROOT ở trên.")

---
## 5. Trích Xuất Metadata JSON Sau Tiền Xử Lý (Post-Preprocessing Metadata Extraction)

Sau khi các video đã được tiền xử lý (cắt khung hình theo TBL và resize về 224x224),
ta cần trích xuất thông tin metadata mới (thời lượng thực tế, số frame mới, độ phân giải mới...)
để lưu thành file JSON. File này sẽ đại diện chính xác cho tập dữ liệu đã sạch.

Việc trích xuất sau tiền xử lý này vô cùng quan trọng vì:
- Độ phân giải đã chuyển từ HD/SD gốc sang chuẩn `224x224` pixel
- Số lượng frames và duration đã giảm đi do TBL loại bỏ tĩnh đầu/cuối
- Giúp đảm bảo dữ liệu đầu vào cho bước EDA và Model Training được nhất quán và chính xác.

In [ ]:
def get_video_metadata(video_path, gloss):
    """
    Trích xuất metadata của một video dùng OpenCV.
    """
    video_filename = os.path.basename(video_path)
    videoid = os.path.splitext(video_filename)[0]
    
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return {
                "videoid": videoid,
                "fps": 0.0,
                "resolution": "0x0",
                "gloss": gloss,
                "num_frames": 0,
                "duration": 0.0,
                "error": "Cannot open video file"
            }
        
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        duration = 0.0
        if fps > 0:
            duration = round(num_frames / fps, 2)
            
        resolution = f"{width}x{height}"
        cap.release()
        
        return {
            "videoid": videoid,
            "fps": round(fps, 2) if fps > 0 else 0.0,
            "resolution": resolution,
            "gloss": gloss,
            "num_frames": num_frames,
            "duration": duration
        }
    except Exception as e:
        return {
            "videoid": videoid,
            "fps": 0.0,
            "resolution": "0x0",
            "gloss": gloss,
            "num_frames": 0,
            "duration": 0.0,
            "error": str(e)
        }

def extract_metadata_after_preprocessing(preprocessed_dir, output_json_path):
    """
    Quét thư mục video đã tiền xử lý và trích xuất metadata thành file JSON.
    """
    if not os.path.exists(preprocessed_dir):
        print(f" Thư mục không tồn tại: {preprocessed_dir}")
        return
        
    print(f"🔍 Đang quét thư mục video đã tiền xử lý: {preprocessed_dir}")
    
    tasks = []
    video_extensions = ('.mp4', '.avi', '.mkv', '.mov', '.flv')
    
    for root, dirs, files in os.walk(preprocessed_dir):
        rel_path = os.path.relpath(root, preprocessed_dir)
        if rel_path == ".":
            continue
            
        gloss = rel_path.split(os.sep)[0]
        
        for file in files:
            if file.lower().endswith(video_extensions):
                video_path = os.path.join(root, file)
                tasks.append((video_path, gloss))
                
    total_videos = len(tasks)
    print(f"Tìm thấy {total_videos} video đã tiền xử lý.")
    
    if total_videos == 0:
        return
        
    results = []
    # Sử dụng ThreadPoolExecutor để tăng tốc độ quét metadata
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(get_video_metadata, path, gloss): (path, gloss) for path, gloss in tasks}
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=total_videos, desc="Trích xuất metadata"):
            try:
                res = future.result()
                results.append(res)
            except Exception as e:
                path, gloss = futures[future]
                print(f"\n Lỗi xử lý {path}: {e}")
                
    print(f"💾 Lưu metadata vào file: {output_json_path}")
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
        
    print("Đã hoàn thành trích xuất metadata cho video sau tiền xử lý!")

### Chạy trích xuất metadata JSON sau tiền xử lý

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi đường dẫn phù hợp với máy của bạn
# ============================================================
# PREPROCESSED_DIR = os.path.join("..", "VSL_FULL_FRONT_CROPPED_TO224x224_V2")
# OUTPUT_JSON_PATH = "preprocessed_vsl_metadata.json"

# extract_metadata_after_preprocessing(PREPROCESSED_DIR, OUTPUT_JSON_PATH)

---
## 6. Tổng Kết

### Sơ đồ quy trình xử lý video & trích xuất metadata:

```
Video gốc (HD, 1080p, thời lượng không đều)
        │
        ▼
┌──────────────────────────────────┐
│  PASS 1: Temporal Boundary       │
│  Localization (TBL)              │
│  - MediaPipe Pose                │
│  - Elbow angle < 160°            │
│  - Gap merging (0.8s)            │
│  - Min duration (0.67s)          │
└──────────────────────────────────┘
        │
        ▼
┌──────────────────────────────────┐
│  PASS 2: Spatial Crop & Resize   │
│  - Bounding Box: 3.6 × shoulder  │
│  - Centered on nose              │
│  - Padding ±0.4s                 │
│  - Output: 224×224 px            │
└──────────────────────────────────┘
        │
        ▼
┌──────────────────────────────────┐
│  POST-PREPROCESSING:             │
│  Metadata JSON Extraction        │
│  - Quét videos đã crop          │
│  - Trích xuất: FPS, Resolution,   │
│    Frames, Duration              │
└──────────────────────────────────┘
        │
        ├──────────────────────────┐
        ▼                          ▼
Video chuẩn hóa 224x224      File JSON metadata mới
```

### Đầu ra:
| Thư mục / File | Nội dung |
|---------|----------|
| `VSL_FULL_FRONT_CROPPED_TO224x224_V2/{gloss}/` | Video đã cắt và resize 224×224 |
| `preprocessed_vsl_metadata.json` | Metadata chi tiết của video đã tiền xử lý |

➡️ **Bước tiếp theo:** Chạy notebook `03_exploratory_data_analysis.ipynb`
để trích xuất keypoints 3D và phân tích dữ liệu.